# Решения: Практика DP 1D: состояния, переходы, восстановление

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import csv


def find_data(name):
    for path in (Path(name), Path("../../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден")


def load_coin_cases():
    rows = []
    with find_data("coin_change_cases.csv").open(encoding="utf-8") as file:
        for row in csv.DictReader(file):
            rows.append((
                row["case_id"],
                int(row["amount"]),
                [int(value) for value in row["coins"].split()],
                int(row["expected_min_coins"]),
            ))
    return rows


def load_grid():
    with find_data("route_cost_grid_4x5.csv").open(encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        return [[int(value) for value in row] for row in reader]


COIN_CASES = load_coin_cases()
ROUTE_GRID = load_grid()
assert len(COIN_CASES) == 5
assert len(ROUTE_GRID) == 4 and len(ROUTE_GRID[0]) == 5


## Урок. 1. Минимум прыжков до адреса

Курьер перемещается на любое число кварталов из `steps`. `dp[position]` хранит минимум перемещений.

In [ ]:
def min_jumps(distance, steps):
    unreachable = distance + 1
    dp = [unreachable] * (distance + 1)
    dp[0] = 0
    for position in range(1, distance + 1):
        for step in steps:
            if position >= step:
                dp[position] = min(dp[position], dp[position - step] + 1)
    return -1 if dp[distance] == unreachable else dp[distance]


assert min_jumps(7, [2, 3]) == 3
assert min_jumps(5, [4]) == -1
assert min_jumps(0, [4]) == 0


## Урок. 2. Сколько оптимальных маршрутов

Вместе с минимумом храните число способов получить этот минимум. Более длинные маршруты не учитывайте.

In [ ]:
def count_min_jump_plans(distance, steps):
    unreachable = distance + 1
    best = [unreachable] * (distance + 1)
    count = [0] * (distance + 1)
    best[0], count[0] = 0, 1
    for position in range(1, distance + 1):
        for step in steps:
            if position < step:
                continue
            candidate = best[position - step] + 1
            if candidate < best[position]:
                best[position] = candidate
                count[position] = count[position - step]
            elif candidate == best[position]:
                count[position] += count[position - step]
    if best[distance] == unreachable:
        return -1, 0
    return best[distance], count[distance]


assert count_min_jump_plans(4, [1, 2, 3]) == (2, 3)
assert count_min_jump_plans(5, [4]) == (-1, 0)


## Урок. 3. Максимальная выручка без соседних смен

Повторите переход «пропустить или взять» без копирования решения прошлой пары.

In [ ]:
def max_profit(values):
    previous_two = 0
    previous_one = 0
    for value in values:
        current = max(previous_one, previous_two + value)
        previous_two, previous_one = previous_one, current
    return previous_one


assert max_profit([6, 7, 1, 30, 8, 2, 4]) == 41
assert max_profit([]) == 0
assert max_profit([9, 1]) == 9


## Урок. 4. Восстановить выбранные смены

Одного оптимального числа недостаточно для диспетчера. Верните индексы смен, которые дают этот максимум.

In [ ]:
def profit_plan(values):
    dp = [0] * (len(values) + 1)
    if values:
        dp[1] = values[0]
    for i in range(2, len(values) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + values[i - 1])
    plan = []
    i = len(values)
    while i > 0:
        if dp[i] == dp[i - 1]:
            i -= 1
        else:
            plan.append(i - 1)
            i -= 2
    return list(reversed(plan))


values = [6, 7, 1, 30, 8, 2, 4]
plan = profit_plan(values)
assert sum(values[i] for i in plan) == 41
assert all(right - left > 1 for left, right in zip(plan, plan[1:]))


## Урок. 5. Маршрут с закрытыми кварталами

Разрешены шаги 1 и 2, но на позиции из `blocked` вставать нельзя. Посчитайте число допустимых маршрутов.

In [ ]:
def routes_avoiding(distance, blocked):
    dp = [0] * (distance + 1)
    dp[0] = 1
    for position in range(1, distance + 1):
        if position in blocked:
            continue
        dp[position] = dp[position - 1]
        if position >= 2:
            dp[position] += dp[position - 2]
    return dp[distance]


assert routes_avoiding(6, {3}) == 4
assert routes_avoiding(3, {1, 2}) == 0
assert routes_avoiding(0, set()) == 1


## Урок. 6. Минимальная стоимость последовательности остановок

На каждой позиции есть стоимость. До позиции можно прийти с одной или двух предыдущих; старт перед первой позицией бесплатный.

In [ ]:
def min_service_cost(costs):
    if not costs:
        return 0
    dp = [0] * (len(costs) + 1)
    dp[1] = costs[0]
    for i in range(2, len(costs) + 1):
        dp[i] = costs[i - 1] + min(dp[i - 1], dp[i - 2])
    return dp[-1]


assert min_service_cost([4, 1, 7, 2, 3]) == 6
assert min_service_cost([5]) == 5
assert min_service_cost([]) == 0


## Урок. 7. Память O(1) вместо таблицы

Для `max_profit` нужны только два предыдущих значения. Реализуйте rolling-вариант и проверьте его против табличного на разных префиксах.

In [ ]:
def max_profit_rolling(values):
    previous_two = 0
    previous_one = 0
    for value in values:
        previous_two, previous_one = previous_one, max(previous_one, previous_two + value)
    return previous_one


sample = [5, 2, 8, 1, 9, 3]
for end in range(len(sample) + 1):
    assert max_profit_rolling(sample[:end]) == max_profit(sample[:end])


## Урок. 8. Эксперимент: как меняется оптимальный план

Изменяйте только доход четвёртой смены от 0 до 40. Найдите первое значение, при котором эта смена входит в восстановленный план.

In [ ]:
base = [6, 7, 1, 0, 8, 2, 4]
threshold = None
for candidate in range(41):
    values = base[:]
    values[3] = candidate
    if 3 in profit_plan(values):
        threshold = candidate
        break
assert threshold is not None
assert 0 <= threshold <= 40
print("порог включения:", threshold)


## Урок. 9. Самостоятельно: тариф с ограничением серии

Нельзя брать три смены подряд. Верните максимальную выручку; состояние должно различать длину текущей серии.

In [ ]:
def max_profit_no_three(values):
    dp = [0] * (len(values) + 1)
    if values:
        dp[1] = values[0]
    if len(values) >= 2:
        dp[2] = values[0] + values[1]
    for i in range(3, len(values) + 1):
        dp[i] = max(
            dp[i - 1],
            dp[i - 2] + values[i - 1],
            dp[i - 3] + values[i - 2] + values[i - 1],
        )
    return dp[-1]


assert max_profit_no_three([5, 6, 7]) == 13
assert max_profit_no_three([5, 6, 7, 8]) == 20
assert max_profit_no_three([]) == 0


## ДЗ. A1. Минимум заправок

До каждой следующей заправки можно проехать расстояние из `jumps`.

In [ ]:
def min_refuels(distance, jumps):
    unreachable = distance + 1
    dp = [unreachable] * (distance + 1)
    dp[0] = 0
    for current in range(1, distance + 1):
        for jump in jumps:
            if current >= jump:
                dp[current] = min(dp[current], dp[current - jump] + 1)
    return -1 if dp[distance] == unreachable else dp[distance]


assert min_refuels(10, [3, 4]) == 3
assert min_refuels(5, [2, 4]) == -1


## ДЗ. A2. План выплат без соседних дней

Верните максимум и один набор индексов дней.

In [ ]:
def payout_plan(values):
    dp = [0] * (len(values) + 1)
    if values:
        dp[1] = values[0]
    for i in range(2, len(values) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + values[i - 1])
    indices = []
    i = len(values)
    while i > 0:
        if dp[i] == dp[i - 1]:
            i -= 1
        else:
            indices.append(i - 1)
            i -= 2
    indices.reverse()
    return dp[-1], indices


values = [8, 4, 5, 9, 3, 1, 7]
best, indices = payout_plan(values)
assert best == 24
assert best == sum(values[i] for i in indices)
assert all(b - a > 1 for a, b in zip(indices, indices[1:]))


## ДЗ. A3. Число маршрутов через обязательную точку

Шаги равны 1 или 2. Посчитайте маршруты от 0 до `distance`, которые обязательно проходят через `checkpoint`.

In [ ]:
def routes_via(distance, checkpoint):
    def ways(length):
        dp = [0] * (length + 1)
        dp[0] = 1
        for i in range(1, length + 1):
            dp[i] = dp[i - 1]
            if i >= 2:
                dp[i] += dp[i - 2]
        return dp[length]
    return ways(checkpoint) * ways(distance - checkpoint)


assert routes_via(6, 3) == 9
assert routes_via(5, 0) == 8


## ДЗ. B1. Ровно k смен

Выберите ровно `k` несоседних смен с максимальной выручкой. Если выбор невозможен, верните `None`.

In [ ]:
def max_exact_k(values, k):
    impossible = -10**9
    skip = [[impossible] * (k + 1) for _ in range(len(values) + 1)]
    take = [[impossible] * (k + 1) for _ in range(len(values) + 1)]
    skip[0][0] = 0
    for i, value in enumerate(values, start=1):
        for chosen in range(k + 1):
            skip[i][chosen] = max(skip[i - 1][chosen], take[i - 1][chosen])
            if chosen > 0 and skip[i - 1][chosen - 1] != impossible:
                take[i][chosen] = skip[i - 1][chosen - 1] + value
    answer = max(skip[-1][k], take[-1][k])
    return None if answer == impossible else answer


assert max_exact_k([6, 7, 1, 30, 8], 2) == 37
assert max_exact_k([5, 4], 2) is None
